# HIP-3 Asset Dashboard

Visualize funding rates, open interest, and trade flow for **HIP-3 builder perps** on Hyperliquid using the [0xArchive SDK](https://pypi.org/project/oxarchive/).

HIP-3 assets are community-created perpetual contracts that trade on Hyperliquid's decentralized exchange. This notebook fetches historical data for any HIP-3 asset and creates:

- **Funding rate time series** with positive/negative shading and annualized view
- **Funding rate distribution** with skew/kurtosis statistics
- **Open interest evolution** overlaid with mark price
- **Trade flow analysis** — taker buy vs sell volume, cumulative delta, and size distribution
- **Mark price with volume and OI** — combined market structure view
- **Hourly volume profile** — time-of-day liquidity patterns

**Requirements:** API key from [0xarchive.io/dashboard](https://0xarchive.io/dashboard). Works with a Free API key: HIP-3 is on every tier, and Free covers the most recent 30 days of history.

**Note:** HIP-3 coins use the `namespace:ticker` format (e.g. `km:NVDA`, `km:GOLD`). Use `hip3.instruments.list()` to see all available assets.

## 1. Setup

In [ ]:
%pip install oxarchive pandas matplotlib seaborn numpy python-dotenv scipy -q

In [ ]:
import os
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats as sp_stats
from oxarchive import Client

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# --- Configuration ---
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env", override=True)

API_KEY = os.environ.get("OXARCHIVE_API_KEY", "your_api_key_here")
if API_KEY == "your_api_key_here":
    raise ValueError("Set OXARCHIVE_API_KEY in .env or as an environment variable")

# HIP-3 coins use namespace:ticker format (e.g. km:NVDA, km:GOLD, km:US500)
COIN = "km:NVDA"
MAX_TRADE_PAGES = 50    # Cap trade fetching (each page = 1000 records)
LOOKBACK_DAYS = 30      # Free covers the most recent rolling 30 days; raise on Build+ (full archive)

# Color palette
COLOR_PRICE = "#f39c12"     # amber  — price line
COLOR_FUNDING = "#3498db"   # blue   — funding rate
COLOR_OI = "#9b59b6"        # purple — open interest
COLOR_BUY = "#2ecc71"       # green  — taker buy (side A)
COLOR_SELL = "#e74c3c"      # red    — taker sell (side B)
COLOR_VOLUME = "#1abc9c"    # teal   — volume bars
COLOR_POS = "#2ecc71"       # green  — positive funding
COLOR_NEG = "#e74c3c"       # red    — negative funding

client = Client(api_key=API_KEY)
end = datetime.now(timezone.utc)

# Auto-detect the earliest available data within the lookback window.
# Free history covers the most recent rolling 30 days (30-day span per
# request); Build and above keep the full archive. Probing from the
# window edge keeps every request inside the Free window while still
# finding the true start for assets newer than the window.
_probe_start = end - timedelta(days=LOOKBACK_DAYS)
_probe = client.hyperliquid.hip3.funding.history(COIN, start=_probe_start, end=end, limit=1)
if not _probe.data:
    raise ValueError(f"No HIP-3 data found for {COIN}")

start = _probe.data[0].timestamp
if not start.tzinfo:
    start = start.replace(tzinfo=timezone.utc)

span = end - start
span_label = (
    f"{span.days}d {span.seconds // 3600}h" if span.days > 0
    else f"{span.seconds // 3600}h {(span.seconds % 3600) // 60}m"
)
print(f"Dashboard for {COIN} (HIP-3)")
print(f"Data available: {start:%Y-%m-%d %H:%M} -> {end:%Y-%m-%d %H:%M} UTC ({span_label})")

## 2. HIP-3 Instrument Info

Fetch metadata for the HIP-3 asset and display the current market snapshot.

In [ ]:
# Fetch HIP-3 instrument details
instrument = client.hyperliquid.hip3.instruments.get(COIN)

print(f"Coin:           {instrument.coin}")
print(f"Namespace:      {instrument.namespace}")
print(f"Ticker:         {instrument.ticker}")
print(f"Mark Price:     ${instrument.mark_price:,.2f}")
print(f"Mid Price:      ${instrument.mid_price:,.2f}")
print(f"Open Interest:  {instrument.open_interest:,.3f}")
print(f"Last Updated:   {instrument.latest_timestamp}")

# Current funding and OI snapshots
current_funding = client.hyperliquid.hip3.funding.current(COIN)
current_oi = client.hyperliquid.hip3.open_interest.current(COIN)

print(f"\nCurrent Funding Rate:  {float(current_funding.funding_rate):.10f}")
print(f"Current Premium:       {float(current_funding.premium):.10f}")
print(f"Current OI:            {float(current_oi.open_interest):,.3f}")
print(f"Oracle Price:          ${float(current_oi.oracle_price):,.2f}")

## 3. Fetch Historical Data

Pull funding rate history, open interest history, and trades using the HIP-3 specific endpoints. The time range is auto-detected from the earliest available data point inside the lookback window. Each dataset is paginated using cursor-based iteration.

In [ ]:
# --- Fetch HIP-3 funding rate history ---
funding_records = []
result = client.hyperliquid.hip3.funding.history(COIN, start=start, end=end, limit=1000)
funding_records.extend(result.data)

while result.next_cursor:
    result = client.hyperliquid.hip3.funding.history(
        COIN, start=start, end=end, cursor=result.next_cursor, limit=1000
    )
    funding_records.extend(result.data)
    print(f"\rFunding: fetched {len(funding_records):,} snapshots...", end="", flush=True)

print(f"\rFunding: {len(funding_records):,} snapshots ({span_label})")

In [ ]:
# --- Fetch HIP-3 open interest history (includes mark_price) ---
oi_records = []
result = client.hyperliquid.hip3.open_interest.history(COIN, start=start, end=end, limit=1000)
oi_records.extend(result.data)

while result.next_cursor:
    result = client.hyperliquid.hip3.open_interest.history(
        COIN, start=start, end=end, cursor=result.next_cursor, limit=1000
    )
    oi_records.extend(result.data)
    print(f"\rOpen Interest: fetched {len(oi_records):,} snapshots...", end="", flush=True)

print(f"\rOpen Interest: {len(oi_records):,} snapshots ({span_label})")

In [ ]:
# --- Fetch HIP-3 trade history ---
trade_records = []
pages = 0
result = client.hyperliquid.hip3.trades.list(COIN, start=start, end=end, limit=1000)
trade_records.extend(result.data)
pages += 1

while result.next_cursor and pages < MAX_TRADE_PAGES:
    result = client.hyperliquid.hip3.trades.list(
        COIN, start=start, end=end, cursor=result.next_cursor, limit=1000
    )
    trade_records.extend(result.data)
    pages += 1
    print(f"\rTrades: fetched {len(trade_records):,} records ({pages} pages)...", end="", flush=True)

truncated = result.next_cursor is not None
suffix = f" (capped at {MAX_TRADE_PAGES} pages)" if truncated else ""
print(f"\rTrades: {len(trade_records):,} records ({span_label}){suffix}")

## 4. Data Processing

Convert raw API records into pandas DataFrames. HIP-3 trades appear twice per fill (side A and B); we deduplicate by `trade_id` keeping only the taker (`crossed=True`) record. Side A = taker buy, side B = taker sell.

In [ ]:
# --- Build funding rate DataFrame ---
funding_df = pd.DataFrame([
    {"timestamp": r.timestamp, "rate": float(r.funding_rate)}
    for r in funding_records
])
funding_df["timestamp"] = pd.to_datetime(funding_df["timestamp"], utc=True)
funding_df = funding_df.set_index("timestamp").sort_index()

# Resample to hourly means
funding_hourly = funding_df["rate"].resample("1h").mean().dropna()

# Convert to 8-hour rate (Hyperliquid reports hourly funding)
funding_8h = funding_hourly * 8

print(f"Funding rate range (8h): [{funding_8h.min():.6f}, {funding_8h.max():.6f}]")
print(f"Mean 8h rate: {funding_8h.mean():.6f}")
print(f"Hours of data: {len(funding_8h)}")

In [ ]:
# --- Build open interest + price DataFrame ---
# OI history includes mark_price, which we use as the price source
# (HIP-3 assets don't have a separate candle endpoint)
oi_df = pd.DataFrame([
    {
        "timestamp": r.timestamp,
        "open_interest": float(r.open_interest),
        "mark_price": float(r.mark_price),
        "oracle_price": float(r.oracle_price),
    }
    for r in oi_records
])
oi_df["timestamp"] = pd.to_datetime(oi_df["timestamp"], utc=True)
oi_df = oi_df.set_index("timestamp").sort_index()

# Resample to hourly
oi_hourly = oi_df["open_interest"].resample("1h").last().dropna()
price_hourly = oi_df["mark_price"].resample("1h").agg(
    open="first", high="max", low="min", close="last"
).dropna()

print(f"OI range: [{oi_hourly.min():,.2f}, {oi_hourly.max():,.2f}]")
print(f"Current OI: {oi_hourly.iloc[-1]:,.2f}")
print(f"Price range: ${price_hourly['low'].min():,.2f} - ${price_hourly['high'].max():,.2f}")
print(f"Hours of data: {len(oi_hourly)}")

In [ ]:
# --- Build trades DataFrame ---
# Each fill produces two records (A + B side). Keep only the taker
# (crossed=True) to get unique fills. Side A = taker buy, B = taker sell.
trades_df = pd.DataFrame([
    {
        "timestamp": r.timestamp,
        "trade_id": r.trade_id,
        "price": float(r.price),
        "size": float(r.size),
        "side": r.side,
        "crossed": r.crossed,
        "notional": float(r.price) * float(r.size),
    }
    for r in trade_records
])

# Deduplicate: keep taker side only
trades_df = trades_df[trades_df["crossed"] == True].copy()
trades_df["timestamp"] = pd.to_datetime(trades_df["timestamp"], utc=True)
trades_df = trades_df.sort_values("timestamp").reset_index(drop=True)

# Label direction: A = taker buy, B = taker sell
trades_df["direction"] = trades_df["side"].map({"A": "Buy", "B": "Sell"})

print(f"Unique fills: {len(trades_df):,}")
print(f"Total notional volume: ${trades_df['notional'].sum():,.2f}")
print(f"Price range: ${trades_df['price'].min():,.2f} - ${trades_df['price'].max():,.2f}")

buy_count = (trades_df["direction"] == "Buy").sum()
sell_count = (trades_df["direction"] == "Sell").sum()
print(f"Taker buys:  {buy_count:,}  |  Taker sells: {sell_count:,}")

## 5. Funding Rate Analysis

Funding rates determine the cost of holding a perpetual position. Positive rates mean longs pay shorts; negative rates mean shorts pay longs. For HIP-3 assets, funding dynamics can be more volatile due to lower liquidity.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

# Top: 8-hour funding rate with positive/negative shading
rate_pct = funding_8h * 100
axes[0].plot(funding_8h.index, rate_pct, color=COLOR_FUNDING,
             linewidth=1.0, alpha=0.9)
axes[0].fill_between(funding_8h.index, 0, rate_pct,
                     where=rate_pct >= 0, color=COLOR_POS, alpha=0.3,
                     interpolate=True, label="Positive (longs pay)")
axes[0].fill_between(funding_8h.index, 0, rate_pct,
                     where=rate_pct < 0, color=COLOR_NEG, alpha=0.3,
                     interpolate=True, label="Negative (shorts pay)")
axes[0].axhline(0, color="white", linewidth=0.5, alpha=0.3)

mean_rate = funding_8h.mean() * 100
axes[0].axhline(mean_rate, color="white", linewidth=0.8, linestyle="--",
                alpha=0.5, label=f"Mean ({mean_rate:.5f}%)")

axes[0].set_ylabel("8-Hour Funding Rate (%)")
axes[0].set_title(f"{COIN} Funding Rate \u2014 {span_label}", fontsize=16)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.4f}%"))
axes[0].legend(loc="upper left", fontsize=10)

# Bottom: mark price overlay for context
axes[1].plot(price_hourly.index, price_hourly["close"], color=COLOR_PRICE,
             linewidth=1.5, alpha=0.9)
axes[1].set_ylabel(f"Mark Price ($)")
axes[1].set_xlabel("Time (UTC)")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

pos_hours = (funding_8h > 0).sum()
neg_hours = (funding_8h < 0).sum()
total_hours = len(funding_8h)
print(f"Positive funding: {pos_hours}/{total_hours} hours ({pos_hours/total_hours*100:.1f}%)")
print(f"Negative funding: {neg_hours}/{total_hours} hours ({neg_hours/total_hours*100:.1f}%)")

In [ ]:
# Annualized funding rate: 8h_rate * 3 * 365
funding_apr = funding_8h * 3 * 365

fig, ax = plt.subplots(figsize=(18, 7))

apr_pct = funding_apr * 100
ax.fill_between(funding_apr.index, 0, apr_pct,
                where=apr_pct >= 0, color=COLOR_POS, alpha=0.4, interpolate=True)
ax.fill_between(funding_apr.index, 0, apr_pct,
                where=apr_pct < 0, color=COLOR_NEG, alpha=0.4, interpolate=True)
ax.plot(funding_apr.index, apr_pct, color=COLOR_FUNDING,
        linewidth=1.0, alpha=0.9)
ax.axhline(0, color="white", linewidth=0.5, alpha=0.3)

mean_apr = funding_apr.mean() * 100
ax.axhline(mean_apr, color="white", linewidth=0.8, linestyle="--",
           alpha=0.5, label=f"Mean APR ({mean_apr:.1f}%)")

ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Annualized Funding Rate (%)")
ax.set_title(f"{COIN} Annualized Funding Rate (APR)", fontsize=16)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
ax.legend(loc="upper left", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"Mean annualized rate: {mean_apr:.2f}%")
print(f"Max annualized rate:  {apr_pct.max():.2f}%")
print(f"Min annualized rate:  {apr_pct.min():.2f}%")

## 6. Funding Rate Distribution

The shape of the funding rate distribution reveals whether the market consistently leans one direction (skew) and whether extreme rates occur more often than a normal distribution would predict (kurtosis).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: 8h rate histogram
rate_vals = funding_8h * 100
axes[0].hist(rate_vals, bins=50, color=COLOR_FUNDING, alpha=0.7, edgecolor="none")
axes[0].axvline(rate_vals.mean(), color="white", linewidth=1.5, linestyle="--", alpha=0.8)
axes[0].axvline(0, color=COLOR_NEG, linewidth=1.0, linestyle="-", alpha=0.4)
axes[0].set_xlabel("8-Hour Funding Rate (%)")
axes[0].set_ylabel("Count (hours)")
axes[0].set_title("8-Hour Rate Distribution", fontsize=14)

skew = sp_stats.skew(funding_8h.values)
kurt = sp_stats.kurtosis(funding_8h.values)
stats_text = (
    f"Mean: {rate_vals.mean():.5f}%\n"
    f"Std:  {rate_vals.std():.5f}%\n"
    f"Skew: {skew:.3f}\n"
    f"Kurt: {kurt:.3f}"
)
axes[0].text(0.97, 0.97, stats_text, transform=axes[0].transAxes,
             verticalalignment="top", horizontalalignment="right",
             fontsize=10, fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="black", alpha=0.5))

# Right: annualized rate histogram
apr_vals = funding_apr * 100
axes[1].hist(apr_vals, bins=50, color=COLOR_FUNDING, alpha=0.7, edgecolor="none")
axes[1].axvline(apr_vals.mean(), color="white", linewidth=1.5, linestyle="--", alpha=0.8)
axes[1].axvline(0, color=COLOR_NEG, linewidth=1.0, linestyle="-", alpha=0.4)
axes[1].set_xlabel("Annualized Rate (%)")
axes[1].set_ylabel("Count (hours)")
axes[1].set_title("Annualized Rate Distribution", fontsize=14)

apr_stats_text = (
    f"Mean: {apr_vals.mean():.2f}%\n"
    f"Std:  {apr_vals.std():.2f}%\n"
    f"Skew: {skew:.3f}\n"
    f"Kurt: {kurt:.3f}"
)
axes[1].text(0.97, 0.97, apr_stats_text, transform=axes[1].transAxes,
             verticalalignment="top", horizontalalignment="right",
             fontsize=10, fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="black", alpha=0.5))

plt.tight_layout()
plt.show()

## 7. Open Interest Analysis

Open interest tracks the total number of outstanding contracts. Rising OI with rising price suggests new longs entering; rising OI with falling price suggests new shorts. Declining OI signals position unwinding.

In [ ]:
fig, ax1 = plt.subplots(figsize=(18, 8))

# OI on left axis
ax1.fill_between(oi_hourly.index, 0, oi_hourly.values,
                 color=COLOR_OI, alpha=0.3)
ax1.plot(oi_hourly.index, oi_hourly.values, color=COLOR_OI,
         linewidth=1.5, alpha=0.9, label="Open Interest")
ax1.set_xlabel("Time (UTC)")
ax1.set_ylabel("Open Interest", color=COLOR_OI)
ax1.tick_params(axis="y", labelcolor=COLOR_OI)

# Mark price on right axis
ax2 = ax1.twinx()
ax2.plot(price_hourly.index, price_hourly["close"], color=COLOR_PRICE,
         linewidth=1.5, alpha=0.9, label="Mark Price")
ax2.set_ylabel(f"Mark Price ($)", color=COLOR_PRICE)
ax2.tick_params(axis="y", labelcolor=COLOR_PRICE)

ax1.set_title(f"{COIN} Open Interest vs Mark Price \u2014 {span_label}", fontsize=16)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
plt.xticks(rotation=45, ha="right")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=12)

plt.tight_layout()
plt.show()

oi_change = oi_hourly.iloc[-1] - oi_hourly.iloc[0]
oi_change_pct = oi_change / oi_hourly.iloc[0] * 100 if oi_hourly.iloc[0] != 0 else 0
print(f"OI at start:  {oi_hourly.iloc[0]:,.2f}")
print(f"OI at end:    {oi_hourly.iloc[-1]:,.2f}")
print(f"OI change:    {oi_change:+,.2f} ({oi_change_pct:+.1f}%)")
print(f"Peak OI:      {oi_hourly.max():,.2f}")
print(f"Trough OI:    {oi_hourly.min():,.2f}")

In [ ]:
# OI rate of change (hourly % change)
oi_pct_change = oi_hourly.pct_change().dropna() * 100

fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharex=True,
                         gridspec_kw={"height_ratios": [1, 1]})

# Top: OI hourly % change
bar_colors = np.where(oi_pct_change >= 0, COLOR_POS, COLOR_NEG)
axes[0].bar(oi_pct_change.index, oi_pct_change.values,
            width=pd.Timedelta("1h"), color=bar_colors, alpha=0.7)
axes[0].axhline(0, color="white", linewidth=0.5, alpha=0.3)
axes[0].set_ylabel("OI Hourly Change (%)")
axes[0].set_title(f"{COIN} Open Interest Rate of Change", fontsize=16)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:+.1f}%"))

# Bottom: funding rate for correlation
axes[1].plot(funding_8h.index, funding_8h * 100, color=COLOR_FUNDING,
             linewidth=1.0, alpha=0.9)
axes[1].fill_between(funding_8h.index, 0, funding_8h * 100,
                     where=funding_8h >= 0, color=COLOR_POS, alpha=0.2, interpolate=True)
axes[1].fill_between(funding_8h.index, 0, funding_8h * 100,
                     where=funding_8h < 0, color=COLOR_NEG, alpha=0.2, interpolate=True)
axes[1].axhline(0, color="white", linewidth=0.5, alpha=0.3)
axes[1].set_ylabel("8h Funding Rate (%)")
axes[1].set_xlabel("Time (UTC)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.4f}%"))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 8. Trade Flow Analysis

Analyze fills to understand buy/sell pressure, volume patterns, and trade size distribution. Cumulative delta (taker buy volume minus taker sell volume) reveals net directional flow.

In [ ]:
# Resample trades into hourly taker buy/sell volume
trades_ts = trades_df.set_index("timestamp")

buy_vol = trades_ts.loc[trades_ts["direction"] == "Buy", "notional"].resample("1h").sum().fillna(0)
sell_vol = trades_ts.loc[trades_ts["direction"] == "Sell", "notional"].resample("1h").sum().fillna(0)

# Align indices
common_idx = buy_vol.index.union(sell_vol.index)
buy_vol = buy_vol.reindex(common_idx, fill_value=0)
sell_vol = sell_vol.reindex(common_idx, fill_value=0)

fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

# Top: buy/sell volume bars
axes[0].bar(buy_vol.index, buy_vol.values, width=pd.Timedelta("1h"),
            color=COLOR_BUY, alpha=0.7, label="Taker Buy Volume")
axes[0].bar(sell_vol.index, -sell_vol.values, width=pd.Timedelta("1h"),
            color=COLOR_SELL, alpha=0.7, label="Taker Sell Volume")
axes[0].axhline(0, color="white", linewidth=0.5, alpha=0.3)
axes[0].set_ylabel("Notional Volume ($)")
axes[0].set_title(f"{COIN} Hourly Trade Volume \u2014 Taker Buy vs Sell", fontsize=16)
axes[0].legend(loc="upper left", fontsize=12)

# Bottom: mark price
axes[1].plot(price_hourly.index, price_hourly["close"], color=COLOR_PRICE,
             linewidth=1.5, alpha=0.9)
axes[1].set_ylabel("Mark Price ($)")
axes[1].set_xlabel("Time (UTC)")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

total_buy = buy_vol.sum()
total_sell = sell_vol.sum()
total_vol = total_buy + total_sell
print(f"Total taker buy volume:  ${total_buy:,.2f} ({total_buy/total_vol*100:.1f}%)")
print(f"Total taker sell volume: ${total_sell:,.2f} ({total_sell/total_vol*100:.1f}%)")
print(f"Net flow (buy-sell):     ${total_buy - total_sell:+,.2f}")

In [ ]:
# Cumulative volume delta (taker buy - taker sell notional)
hourly_delta = buy_vol - sell_vol
cum_delta = hourly_delta.cumsum()

fig, ax1 = plt.subplots(figsize=(18, 8))

# Cumulative delta on left axis
ax1.fill_between(cum_delta.index, 0, cum_delta.values,
                 where=cum_delta >= 0, color=COLOR_BUY, alpha=0.3, interpolate=True)
ax1.fill_between(cum_delta.index, 0, cum_delta.values,
                 where=cum_delta < 0, color=COLOR_SELL, alpha=0.3, interpolate=True)
ax1.plot(cum_delta.index, cum_delta.values, color="#ecf0f1",
         linewidth=1.5, alpha=0.9, label="Cumulative Delta")
ax1.axhline(0, color="white", linewidth=0.5, alpha=0.3)
ax1.set_ylabel("Cumulative Delta ($)", color="#ecf0f1")
ax1.tick_params(axis="y", labelcolor="#ecf0f1")

# Mark price on right axis
ax2 = ax1.twinx()
ax2.plot(price_hourly.index, price_hourly["close"], color=COLOR_PRICE,
         linewidth=1.5, alpha=0.9, label="Mark Price")
ax2.set_ylabel(f"Mark Price ($)", color=COLOR_PRICE)
ax2.tick_params(axis="y", labelcolor=COLOR_PRICE)

ax1.set_xlabel("Time (UTC)")
ax1.set_title(f"{COIN} Cumulative Volume Delta vs Mark Price", fontsize=16)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
plt.xticks(rotation=45, ha="right")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: notional size histogram by direction
for direction, color in [("Buy", COLOR_BUY), ("Sell", COLOR_SELL)]:
    subset = trades_df.loc[trades_df["direction"] == direction, "notional"]
    axes[0].hist(subset, bins=80, alpha=0.6, color=color, label=direction,
                 log=True, edgecolor="none")

axes[0].set_xlabel("Notional Value ($)")
axes[0].set_ylabel("Count (log scale)")
axes[0].set_title("Trade Size Distribution", fontsize=14)
axes[0].legend()
axes[0].set_xlim(0, trades_df["notional"].quantile(0.99))

# Right: summary stats table
buy_stats = trades_df.loc[trades_df["direction"] == "Buy", "notional"].describe(
    percentiles=[0.5, 0.75, 0.95, 0.99]
)
sell_stats = trades_df.loc[trades_df["direction"] == "Sell", "notional"].describe(
    percentiles=[0.5, 0.75, 0.95, 0.99]
)

stats_data = pd.DataFrame({"Buy": buy_stats, "Sell": sell_stats})
stats_display = stats_data.loc[["count", "mean", "50%", "75%", "95%", "99%", "max"]]

axes[1].axis("off")
table = axes[1].table(
    cellText=[[f"{v:,.2f}" for v in row] for row in stats_display.values],
    rowLabels=["Count", "Mean $", "Median $", "P75 $", "P95 $", "P99 $", "Max $"],
    colLabels=stats_display.columns,
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)
axes[1].set_title("Notional Value Statistics by Direction ($)", fontsize=14, pad=20)

plt.tight_layout()
plt.show()

## 9. Mark Price, Volume & Open Interest

Combined market structure view with mark price (from OI snapshots), hourly trade volume, and open interest.

In [ ]:
# Hourly trade volume from trades
hourly_vol_total = trades_ts["notional"].resample("1h").sum().fillna(0)

fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1, 1]})

# Top: mark price with high/low range
axes[0].fill_between(price_hourly.index, price_hourly["low"], price_hourly["high"],
                     color=COLOR_PRICE, alpha=0.15, label="H/L Range")
axes[0].plot(price_hourly.index, price_hourly["close"], color=COLOR_PRICE,
             linewidth=1.5, alpha=0.9, label="Close")
axes[0].set_ylabel("Mark Price ($)")
axes[0].set_title(f"{COIN} Price, Volume & Open Interest \u2014 {span_label}", fontsize=16)
axes[0].legend(loc="upper left", fontsize=10)

# Middle: volume bars colored by net direction
net_dir = (buy_vol - sell_vol).reindex(hourly_vol_total.index, fill_value=0)
vol_colors = np.where(net_dir >= 0, COLOR_BUY, COLOR_SELL)
axes[1].bar(hourly_vol_total.index, hourly_vol_total.values,
            width=pd.Timedelta("1h"), color=vol_colors, alpha=0.7)
axes[1].set_ylabel("Volume ($)")

# Bottom: open interest
axes[2].fill_between(oi_hourly.index, 0, oi_hourly.values,
                     color=COLOR_OI, alpha=0.3)
axes[2].plot(oi_hourly.index, oi_hourly.values, color=COLOR_OI,
             linewidth=1.2, alpha=0.9)
axes[2].set_ylabel("Open Interest")
axes[2].set_xlabel("Time (UTC)")
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 10. Hourly Volume Profile

When is this HIP-3 asset most actively traded? The hour-of-day breakdown reveals liquidity patterns.

In [ ]:
trades_df["hour"] = trades_df["timestamp"].dt.hour

hourly_profile = trades_df.groupby("hour").agg(
    total_notional=("notional", "sum"),
    trade_count=("notional", "count"),
    avg_size=("notional", "mean"),
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Volume by hour
axes[0].bar(hourly_profile.index, hourly_profile["total_notional"],
            color=COLOR_VOLUME, alpha=0.8)
axes[0].set_xlabel("Hour of Day (UTC)")
axes[0].set_ylabel("Total Notional ($)")
axes[0].set_title("Volume by Hour", fontsize=14)
axes[0].set_xticks(range(0, 24, 2))

# Trade count by hour
axes[1].bar(hourly_profile.index, hourly_profile["trade_count"],
            color=COLOR_FUNDING, alpha=0.8)
axes[1].set_xlabel("Hour of Day (UTC)")
axes[1].set_ylabel("Number of Fills")
axes[1].set_title("Fill Count by Hour", fontsize=14)
axes[1].set_xticks(range(0, 24, 2))

# Average trade size by hour
axes[2].bar(hourly_profile.index, hourly_profile["avg_size"],
            color=COLOR_OI, alpha=0.8)
axes[2].set_xlabel("Hour of Day (UTC)")
axes[2].set_ylabel("Avg Notional ($)")
axes[2].set_title("Avg Fill Size by Hour", fontsize=14)
axes[2].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

peak_hour = hourly_profile["total_notional"].idxmax()
quiet_hour = hourly_profile["total_notional"].idxmin()
print(f"Most active hour (UTC):  {peak_hour:02d}:00")
print(f"Least active hour (UTC): {quiet_hour:02d}:00")

## 11. Summary Statistics

In [ ]:
print(f"{'='*65}")
print(f"  {COIN} HIP-3 ASSET DASHBOARD \u2014 {span_label}")
print(f"  {start:%Y-%m-%d %H:%M} to {end:%Y-%m-%d %H:%M} UTC")
print(f"{'='*65}")

print(f"\n  Data Points")
print(f"  {'-'*45}")
print(f"  Funding snapshots:      {len(funding_records):>12,}")
print(f"  OI snapshots:           {len(oi_records):>12,}")
print(f"  Unique fills:           {len(trades_df):>12,}")

print(f"\n  Mark Price")
print(f"  {'-'*45}")
print(f"  Open:                   ${price_hourly['open'].iloc[0]:<12.2f}")
print(f"  Close:                  ${price_hourly['close'].iloc[-1]:<12.2f}")
price_change = price_hourly['close'].iloc[-1] - price_hourly['open'].iloc[0]
price_change_pct = price_change / price_hourly['open'].iloc[0] * 100
print(f"  Change:                 ${price_change:<+12.2f} ({price_change_pct:+.2f}%)")
print(f"  High:                   ${price_hourly['high'].max():<12.2f}")
print(f"  Low:                    ${price_hourly['low'].min():<12.2f}")

print(f"\n  Funding Rate (8h)")
print(f"  {'-'*45}")
print(f"  Mean:                   {funding_8h.mean()*100:>11.5f}%")
print(f"  Std:                    {funding_8h.std()*100:>11.5f}%")
print(f"  Max:                    {funding_8h.max()*100:>11.5f}%")
print(f"  Min:                    {funding_8h.min()*100:>11.5f}%")
print(f"  Skew:                   {skew:>11.3f}")
print(f"  Kurtosis:               {kurt:>11.3f}")
print(f"  Mean APR:               {funding_apr.mean()*100:>11.2f}%")
print(f"  Positive hours:         {pos_hours:>8}/{total_hours} ({pos_hours/total_hours*100:.1f}%)")

print(f"\n  Open Interest")
print(f"  {'-'*45}")
print(f"  Start:                  {oi_hourly.iloc[0]:>12,.2f}")
print(f"  End:                    {oi_hourly.iloc[-1]:>12,.2f}")
print(f"  Change:                 {oi_change:>+12,.2f} ({oi_change_pct:+.1f}%)")
print(f"  Peak:                   {oi_hourly.max():>12,.2f}")
print(f"  Trough:                 {oi_hourly.min():>12,.2f}")

print(f"\n  Trade Flow")
print(f"  {'-'*45}")
print(f"  Unique fills:           {len(trades_df):>12,}")
print(f"  Total volume:           ${trades_df['notional'].sum():>12,.2f}")
print(f"  Avg fill size:          ${trades_df['notional'].mean():>12,.2f}")
print(f"  Median fill size:       ${trades_df['notional'].median():>12,.2f}")
print(f"  Taker buy volume:       ${total_buy:>12,.2f} ({total_buy/total_vol*100:.1f}%)")
print(f"  Taker sell volume:      ${total_sell:>12,.2f} ({total_sell/total_vol*100:.1f}%)")
print(f"  Net delta:              ${total_buy - total_sell:>+12,.2f}")
print(f"{'='*65}")

In [ ]:
client.close()
print("Client closed. Done!")